Train Validation Test split

In [ ]:
human = pd.read_csv('/content/drive/MyDrive/experiment1/dataset/train/human_1000.csv')
ai = pd.read_csv('/content/drive/MyDrive/experiment1/dataset/train/ai_1000.csv')
refine = pd.read_csv('/content/drive/MyDrive/experiment1/dataset/train/refine_1000.csv')

human['label'] = 'human'
ai['label'] = 'ai_generated'
refine['label'] = 'refine'

# Train: 0-699
train = pd.concat([human.iloc[:700], ai.iloc[:700], refine.iloc[:700]], ignore_index=True)

# Val: 700-849
val = pd.concat([human.iloc[700:850], ai.iloc[700:850], refine.iloc[700:850]], ignore_index=True)

# Test: 850-end
test = pd.concat([human.iloc[850:], ai.iloc[850:], refine.iloc[850:]], ignore_index=True)

train.to_csv('/content/drive/MyDrive/experiment1/dataset/train/train_2100.csv', index=False)
val.to_csv('/content/drive/MyDrive/experiment1/dataset/validation/val_450.csv', index=False)
test.to_csv('/content/drive/MyDrive/experiment1/dataset/test/test_450.csv', index=False)

print(f"Train: {len(train)}, Val: {len(val)}, Test: {len(test)}")
print(train['label'].value_counts())
print(val['label'].value_counts())
print(test['label'].value_counts())

extract 3 features and classification

In [ ]:
!pip install sentence-transformers -q

import pandas as pd
import numpy as np
import torch
import math
import re
from transformers import GPT2LMHeadModel, GPT2Tokenizer
from sentence_transformers import SentenceTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from scipy.spatial.distance import cosine

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load Data
# ============================================================
train_df = pd.read_csv('/content/drive/MyDrive/experiment1/dataset/train/train_2100.csv')
val_df = pd.read_csv('/content/drive/MyDrive/experiment1/dataset/validation/val_450.csv')
test_df = pd.read_csv('/content/drive/MyDrive/experiment1/dataset/test/test_450.csv')

# Load Models for Feature Extraction
# ============================================================
# GPT-2 for Perplexity
gpt2_model = GPT2LMHeadModel.from_pretrained('gpt2').to(device)
gpt2_tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
gpt2_model.eval()

# Sentence-BERT for Semantic Shift
sbert = SentenceTransformer('all-MiniLM-L6-v2', device=device)

print("Models loaded")

# Feature Extraction Functions
# ============================================================

# Feature 1: Perplexity (PPL)
# Lower PPL = more predictable = more likely AI
def calc_ppl(text):
    try:
        inputs = gpt2_tokenizer(
            text, return_tensors='pt',
            truncation=True, max_length=512
        ).to(device)
        with torch.no_grad():
            outputs = gpt2_model(**inputs, labels=inputs['input_ids'])
        return math.exp(outputs.loss.item())
    except:
        return 0.0

# Feature 2: Burstiness
# Higher burstiness = more variance in sentence length = more likely human
def calc_burstiness(text):
    sentences = re.split(r'[;.\n]', text)
    lengths = [len(s.split()) for s in sentences if s.strip()]
    if len(lengths) > 1:
        return np.std(lengths)
    return 0.0

# Feature 3: Semantic Shift
# Average cosine distance between consecutive sentence embeddings
# Higher shift = more topic jumps = more likely human
def calc_semantic_shift(text):
    sentences = re.split(r'[;.\n]', text)
    sentences = [s.strip() for s in sentences if len(s.strip()) > 10]
    if len(sentences) < 2:
        return 0.0
    embeddings = sbert.encode(sentences)
    distances = []
    for i in range(len(embeddings) - 1):
        dist = cosine(embeddings[i], embeddings[i + 1])
        distances.append(dist)
    return np.mean(distances)

#  Extract Features
# ============================================================
def extract_features(df, name="data"):
    ppls, bursts, shifts = [], [], []
    total = len(df)

    for i, text in enumerate(df['claim1']):
        text = str(text)
        ppls.append(calc_ppl(text))
        bursts.append(calc_burstiness(text))
        shifts.append(calc_semantic_shift(text))

        if i % 100 == 0:
            print(f"  {name}: {i}/{total}")

    df = df.copy()
    df['ppl'] = ppls
    df['burstiness'] = bursts
    df['semantic_shift'] = shifts
    return df

print("Extracting train features...")
train_feat = extract_features(train_df, "train")

print("Extracting val features...")
val_feat = extract_features(val_df, "val")

print("Extracting test features...")
test_feat = extract_features(test_df, "test")

# Save features to Drive
train_feat.to_csv('/content/drive/MyDrive/experiment1/dataset/train_features.csv', index=False)
val_feat.to_csv('/content/drive/MyDrive/experiment1/dataset/val_features.csv', index=False)
test_feat.to_csv('/content/drive/MyDrive/experiment1/dataset/test_features.csv', index=False)
print("Features saved")

# Check Feature Distributions
# ============================================================
print("\n=== Feature distributions by label ===\n")
for feat in ['ppl', 'burstiness', 'semantic_shift']:
    print(f"--- {feat} ---")
    print(train_feat.groupby('label')[feat].describe()[['mean', 'std', 'min', 'max']])
    print()

# Random Forest Classification
# ============================================================
feature_cols = ['ppl', 'burstiness', 'semantic_shift']

X_train = train_feat[feature_cols]
y_train = train_feat['label']

X_val = val_feat[feature_cols]
y_val = val_feat['label']

X_test = test_feat[feature_cols]
y_test = test_feat['label']

clf = RandomForestClassifier(n_estimators=200, random_state=42)
clf.fit(X_train, y_train)

# Validation
val_preds = clf.predict(X_val)
print(f"Val Accuracy: {accuracy_score(y_val, val_preds):.4f}\n")

# Test
test_preds = clf.predict(X_test)
print(f"Test Accuracy: {accuracy_score(y_test, test_preds):.4f}\n")
print("Classification Report:")
print(classification_report(
    y_test, test_preds,
    target_names=['human', 'ai_generated', 'refine']
))
print("Confusion Matrix:")
print(confusion_matrix(y_test, test_preds))

# Feature Importance
print("\nFeature Importance:")
for name, imp in zip(feature_cols, clf.feature_importances_):
    print(f"  {name}: {imp:.4f}")